# 自动微分 (Autograd)

---
## Autograd怎样记录运算
#### 先认识几个词
- **自动微分 (automatic differentiation / Autograd)**: 框架根据你执行的张量运算, 自动计算导数. 
- **计算图 (computational graph)**: 记录"哪些输入经过哪些运算, 得到哪个结果"的关系图. 箭头按计算方向走. 
- **```requires_grad=True```**: 告诉PyTorch: "后续运算要为这个张量保留求导所需的信息." 它不是立刻算出导数.
- **```grad_fn```**: 结果张量上的一个属性, 记录了产生它的运算对应的反向计算信息. 它不是导数的数值本身. 

可以把计算图想成一张运算流程图: 输入经过乘法, 加法等操作, 一步一步得到结果. **有向无环 (directed acyclic graph, DAG)** 是它正式叫法: "有向"表示箭头有方向; "无环"表示沿箭头不会绕回原处.




#### 看一个最小例子

In [ ]:
import torch

x = torch.arange(4.0)
x.requires_grad_(True)

y = 2 * torch.dot(x, x)

print(x)
print(y)
print(x.requires_grad)
print(x.grad_fn)
print(y.grad_fn)

这里```x```是```[0., 1., 2., 3.]```, 点积结果是```14```, 所以```y```是```28```

计算图可以简化成:
```
x ──┬──> 点积 dot(x, x) ──> 乘以 2 ──> y
    └──> 点积 dot(x, x)
```

因为```x.requires_grad```是```True```, PyTorch在前向计算, 算出```y```的同时, 也记录了这些运算. ```y.grad_fn```会显示产生```y```的运算信息. D2L用这个例子展示了从设置```requires_grad```到形成运算记录的过程; PyTorch官方教程也说明了张量运算会形成计算图, 并把相应的反向函数记录在结果的```grad_fn```中. 

注意: 上面还没有调用```.backward()```, 所以只是建立了运算记录, 尚未执行反向求导.

#### 放进神经网络里看
PyTorch官方教程用一个简单模型距离: 输入```x```和参数```w```, ```b```经过运算得到预测, 再和正确答案比较, 得到```loss```(损失值). 

```
输入 x ──────┐
            ├──> 线性运算 z ──> loss
参数 w、b ───┘                   ↑
正确答案 y ──────────────────────┘
```

如果我们想知道应该怎样调整```w```和```b```来改变损失, 就需要追踪损失是怎样依赖它们的, 因此要对```w```和```b```开启梯度追踪. 输入和正确答案通常是数据, 不需要求它们的梯度.

---
## 用```.backword()```求梯度, 再从```.grad```读取结果

#### 两个新名字
- ```.backward()```: 从结果开始, 沿计算图反向计算它对输入的导数. 
- ```.grad```: 张量的属性, 用来查看反向计算得到的梯度. 
可以先把流程记成: ```requires_grad=True```让PyTorch**记录运算**; ```.backward()```让它**计算梯度**; ```.grad```让你**查看梯度**. 

#### 先看一个数

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x

y.backward()

print(y)       # tensor(15., grad_fn=<...>)
print(x.grad)  # tensor(8.)

我们手动算一下: 
```
y = x² + 2x
dy/dx = 2x + 2
x = 3 时，dy/dx = 8
```

所以```.backward()```后, ```x.grad```是```8```. 注意: ```y```的值是```15```, 而```x.grad```是```8```, 它们分别表示函数结果和函数再```x=3```处的导数, 含义不同.


#### 输入是向量时
D2L还用向量```x = [0, 1, 2, 3]```演示: 

In [ ]:
x = torch.arange(4.0, requires_grad=True)
y = 2 * torch.dot(x, x)

y.backward()

print(y)       # tensor(28., grad_fn=<...>)
print(x.grad)  # tensor([0., 4., 8., 12.])

这里$y = 2(x_0^2 + x_1^2 + x_2^2 + x_3^2)$, 对每个分量分别求偏导, 得到```4x```, 所以梯度是```[0, 4, 8, 12]```. 标量结果对向量输入求梯度, 得到的梯度向量与输入形状相同. 

---
## 叶子张量与计算图生命周期

- **叶子张量 (leaf tensor)** 可以先理解为求导计算起点; 
- **非叶子张量 (non-leaf tensor)** 则是由已追踪的运算产生的结果. 

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)
u = x * x
y = 3 * u

<table>
<tr><th>张量</th><th>怎样产生</th><th>is_leaf</th><th>grad_fn</th></tr>
<tr><td>x</td><td>直接创建起点</td><td>True</td><td>None</td></tr>
<tr><td>u</td><td>通过 x*x 得到</td><td>False</td><td>有</td></tr>
<tr><td>y</td><td>通过 3*u 得到</td><td>False</td><td>有</td></tr>
</table>

这里讨论的是```requires_grad=True```的张量. 另一个约定是:```requires_grad=False```的张量也被视为叶子张量.

默认调用```y.backward()```后, PyTorch会把梯度保存在参与计算, 需要梯度的叶子张量上. 所以```x.grad```是```18```, 而`u.grad`默认仍是`None`. `None`表示没有保存到这个属性里, 不能理解成导数是零. 实际的`dy/du`是`3`. 

如果也想查看中间结果`u`的梯度, 可以在反传前调用`u.retain_grad()`, 意思是"请把它的梯度也保存下来`: 

In [ ]:
u.retain_grad()
y.backward()

print(x.grad)  # tensor(18.)
print(u.grad)  # tensor(3.)

这不会改变求导结果, 只是让你能够从`u.grad`读到它.

接着看计算图的生命周期. 前向计算时, 某些运算需要保存数值, 供反向求导使用. 默认反传完成后, 这些保存的数值会被释放, 以节省内存. 因此下面这个例子第二次反传会报错. 

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = a ** 2

b.backward()
b.backward()  # RuntimeError

`a`, `b`的数值仍然存在; 缺少的是再次执行反传所需的已保存信息. 如果确实要在同一段计算上反复上传两次, 从创建`a`, `b`开始重新运行, 并把最后两行改为: 

In [ ]:
b.backward(retain_graph=True)
b.backward()

第一次的`retain_graph=True`会保留后续反传所需信息. 日常训练通常每轮重新执行前向计算, 所以一般不需要开启它. 



<table>
<tr><th>用法</th><th>保留什么</th></tr>
<tr><td>u.retain_grad()</td><td>将中间张量的梯度保存到u.grad</td></tr>
<tr><td>b.backward(retain_graph=True)</td><td>保留再次反传所需的计算图信息</td></tr>
</table>

---
## 梯度为什么会累积, 以及怎样清零. 

**梯度累积 (gradient accumulation)**: 再次计算出梯度时, PyTorch默认把新梯度加到已有的`.grad`上. 它允许多个计算结果对同一个参数的梯度贡献相加. 

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)

y1 = x ** 2
y1.backward()
print(x.grad)  # tensor(6.)

y2 = 2 * x
y2.backward()
print(x.grad)  # tensor(8.)

第一次, $y1=x^2$的导数是$2x$, 在$x=3$时等于$6$. \
第二次, $y2=2x$的导数是$2$, 已有的梯度是$6$, 再加上这次的$2$, 等于$8$. 

这里每次都重新执行运算, 建立新的计算图, 因而不需要`retain_graph=True`; 但是两张图使用的是同一个`x`, 它的`.grad`会继续累积.

In [ ]:
x.grad.zero_()
print(x.grad)  # tensor(0.)

y3 = x ** 2
y3.backward()
print(x.grad)  # tensor(6.)

`zero_()`表示把已有张量里的数值直接改成零, 所以`x.grad.zero_()`清除的是保存梯度. `x`本身仍然是3. 这里调用时, `x.grad`已经是梯度张量; 如果它还是`None`, 就不能调用`.zero_()`.

<table>
<tr><th>操作</th><th>作用</th></tr>
<tr><td>重新执行前向运算</td><td>建立新的计算图</td></tr>
<tr><td>.backward()</td><td>计算梯度, 并累加到.grad</td></tr>
<tr><td>.grad.zero_()</td><td>清楚已经保存的梯度</td></tr>
</table>

训练时,如果希望每次参数只使用当前批次的数据所产生的梯度, 就要先清楚之前的梯度. 之后学习优化器的时候, `optimizer.zero_grad()`会帮我们统一处理参数的梯度

---
## 输出求和和gradient参数

先看这个例子: 

In [ ]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x * x

此时`y = [4, 9]`, 有两个输出. 直接调用y.backward会报错, 因为我们还没有指定: 这两个输出怎样共同参与这次反传. 

最容易理解的办法是先求和, 让结果变成一个标量: 

In [ ]:
import torch

x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x * x

loss = y.sum()
loss.backward()

print(loss)    # tensor(13., grad_fn=<...>)
print(x.grad)  # tensor([4., 6.])

这里实际求的是: 
```
loss = x₀² + x₁²
梯度 = [2x₀, 2x₁] = [4, 6]
```

还可以把上面的`loss.backward()`及其前一行, 替换为:

In [ ]:
y.backward(gradient=torch.ones_like(y))

`torch.ones_like(y)`会创建一个形状, 数据类型和设备与`y`相同的全1张量, 这里就是`[1, 1]`. 它传入的意思可以理解为: 每个输出各贡献一份, 相当于对$y_0 + y_1$求导. 

如果传入的数字不同, 就可以理解为**加权求和 (weighted sum)**: 每个输出先乘一个系数, 再相加.

In [ ]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x * x

y.backward(gradient=torch.tensor([1.0, 2.0]))

print(x.grad)  # tensor([4., 12.])

这次相当于对下面这个标量求导: 
```
L = 1 × y₀ + 2 × y₁
  = x₀² + 2x₁²

梯度 = [2x₀, 4x₁] = [4, 12]
```

所以, 传入的`[1, 2]`表示`L`对各个输出的梯度; 它会通过链式法则继续传回`x`. 它不会直接成为`x.grad`. 

补充一个精确的小细节: 实数输出只有一个元素时, 可以省略`gradient`, 即使它形状是`[1]`; 这里需要指定, 是因为`y`有两个元素. 

---
## 雅可比矩阵与雅可比乘积

**雅可比矩阵 (Jacobian matrix)**: 把每个输出对每个输入的偏导数排列起来: 每一行对应一个输出, 每一列对应一个输入

比如有两个输入, 两个输出: 
```
x = [2, 3]
y₀ = x₀²
y₁ = x₀ × x₁
```

它们的偏导数是: 
<table>
<tr><th>输出</th><th>对 x₀ 求偏导</th><th>对 x₁ 求偏导</th></tr>
<tr><td>y₀ = x₀²</td><td>2x₀ = 4</td><td>0</td></tr>
<tr><td>y₁ = x₀x₁</td><td>x₁ = 3</td><td>x₀ = 2</td></tr>
</table>

所以, 在`x = [2, 3]`处, 雅可比矩阵就是: 
```
J = [[4, 0],
     [3, 2]]
```

如果向`y.backward(...)`传入`v = [3, 1]`, 它相当于计算标量`L = 3y₀ + y₁`对`x`的梯度. 这个运算叫**向量-雅可比乘积 (vector-Jacobian product, VJP)**, 官方写成$V^TJ$. `.backward(v)`直接计算这个乘积, 无须先把完整的雅可比矩阵保存出来. 

接着看官方矩阵例子. 先解释三个写法: `torch.eye(4, 5)`创建一个4行5列, 主对角线为1,
 其余位置为0的矩阵; 1`.pow(2)`是逐元素平方; `.t()`是二维矩阵转置. 

In [ ]:
import torch

inp = torch.eye(4, 5, requires_grad=True)
out = (inp + 1).pow(2).t()
v = torch.ones_like(out)

out.backward(v, retain_graph=True)
print(inp.grad)

out.backward(v, retain_graph=True)
print(inp.grad)

inp.grad.zero_()
out.backward(v)
print(inp.grad)

---
## 分离计算与关闭梯度追踪

`detach()`可以理解成: 取出一个张量当前的数值, 得到一个与原计算历史分离的张量. 之后通过这个新张量反传时, 梯度不会沿着原来的路径继续传回去.

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x * x
u = y.detach()
z = u * x

z.backward()

print(x.grad)           # tensor(9.)
print(y.requires_grad)  # True
print(u.requires_grad)  # False

虽然`u`的数值来自`x²`, 但在计算`z`的梯度时, PyTorch把`u`当作常数. 此时`u = 9`, 所以相当于: 
```
z = 9 × x
dz/dx = 9
```
如果直接写`z = y * x`, 则会沿`y`的计算路径继续求导, 相当于对`x³`求导, 在`x = 3`时得到`27`. 相同的前向数值, 可以因为梯度路径不同而得到不同的梯度. 

原来的`y`仍然保留自己的计算历史.

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x * x
u = y.detach()
z = u * x

z.backward()

print(x.grad)           # tensor(9.)
print(y.requires_grad)  # True
print(u.requires_grad)  # False

x.grad.zero_()
y.backward()
print(x.grad)  # tensor(6.)

仍能算出`y=x²`对`x`的导数.

另外, `detach()`返回的新张量与原张量共享底层数据; 它没有复制一份独立数据. 

`torch.no_grad()`则让代码中的普通张量运算暂时不记录求导历史. 它常用于**推理 (inference)**, 也就是让训练好的模型计算预测结果, 而不需要反向求导. 

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

with torch.no_grad():
    a = x * x

b = x * x

print(a.requires_grad)  # False
print(b.requires_grad)  # True
print(x.requires_grad)  # True

`no_grad()`影响代码块内的运算记录, `x`自己的标志仍然是`True`. 离开代码块后, 正常的梯度追踪恢复; 但不会补回`a`在代码块缺失的计算历史. 不记录这些历史通常能节省内存和计算开销.

还有一种需求是**冻结参数 (freeze parameters)**: 让某些参数在训练中保持不变. Autograd这一侧, 可以对叶子参数设置`w.requires_grad_(False)`, 停止为它计算新梯度. 如果其他输入仍需要梯度, 相关运算依然可以被追踪.

<table>
<tr><th>写法</th><th>影响范围</th></tr>
<tr><td>u = y.detach()</td><td>新张量u与y的计算历史分离</td></tr>
<tr><td>with torch.no_grad(): </td><td>代码块内的运算不记录求导历史</td></tr>
<tr><td>w.requires_grad_(False)</td><td>停止为叶子参数w计算新梯度</td></tr>
</table>

---
## Python控制流中的梯度计算
**控制流 (control flow)** 就是决定程序执行哪些语句, 执行多少次的规则, 例如`if`条件分支, `while`循环.

**动态图 (dynamic computational graph)**: 每次执行前向计算时, PyTorch根据当次实际执行的张量运算建立计算图. 因此, 不同输入可以走不同分支, 循环不同次数, 之后仍然沿实际路径求导.

下面是D2L的原始函数, 我们用确定的输入代替随机输入, 方便手算:

In [ ]:
import torch

def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

a = torch.tensor(3.0, requires_grad=True)
d = f(a)
d.backward()

print(d)       # 数值为 1536
print(a.grad)  # tensor(512.)

这里`a`, `b`都是标量, 所以`b.norm()`等于其绝对值, `b.sum()`是它自身.

当`a=3`时, 先得到`b=6`, 循环不断翻倍:
```
6 → 12 → 24 → 48 → 96 → 192 → 384 → 768 → 1536
```
最后一次翻倍后, 1536 < 1000为False, 循环停止. 因为`b`为正, 程序执行`c=b`, 于是得到`d=1536`.

在这次执行路径中, 最开始乘以2, 循环又乘了8次2, 所以:
```
d = 2⁹ × a = 512a
dd/da = 512
```